<a href="https://colab.research.google.com/github/aisutd/vibecheck/blob/generative-model/generatemodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
from pathlib import Path

# Keep the data folder in one variable so we can change it later.
DATA_DIR = Path("/content")

# Read a JSONL file: each nonempty line is one JSON example.
def load_jsonl(filename):
    with open(DATA_DIR / filename, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

# Load the training and validation examples into Python lists.
train_data = load_jsonl("sft_train.jsonl")
val_data = load_jsonl("sft_val.jsonl")

# Check how many examples were loaded.
print("Training rows:", len(train_data))
print("Validation rows:", len(val_data))

# Display the first training example with readable indentation.
print(json.dumps(train_data[0], indent=2))

Training rows: 443
Validation rows: 49
{
  "messages": [
    {
      "role": "user",
      "content": "You are an expert security engineer. Fix all security vulnerabilities and hardcoded secrets in the code below.\n\nFILE PATH: nginx.conf\nFILE NAME: nginx.conf\n\nSECURITY ISSUES TO FIX:\n=== OpenGrep Security Findings ===\n- [WARNING] Rule: generic.nginx.security.alias-path-traversal.alias-path-traversal\n  Line 13: The alias in this location block is subject to a path traversal because the location path does not end in a path separator (e.g., '/'). To fix, add a path separator to the end of the path.\n=== OpenGrep Security Findings ===\n- [WARNING] Rule: generic.nginx.security.alias-path-traversal.alias-path-traversal\n  Line 18: The alias in this location block is subject to a path traversal because the location path does not end in a path separator (e.g., '/'). To fix, add a path separator to the end of the path.\nREPAIR INSTRUCTIONS:\n1. Fix ALL listed security issues while mainta

In [ ]:
# Install the library used to load the tokenizer.
%pip install -q transformers

In [ ]:
import os

# Switch to an existing working folder.
os.chdir("/content")

print("Current folder:", os.getcwd())

Current folder: /content


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# Load the tokenizer belonging to the assigned model.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Get the conversation from the first training example.
messages = train_data[0]["messages"]

# Show how the tokenizer formats this conversation for training.
formatted = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)
print(formatted)

## Return a plain list of token IDs so len() counts tokens.
tokens = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    return_dict=False
)

print("\nToken count:", len(tokens))
print("Within 4096-token limit:", len(tokens) <= 4096)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
You are an expert security engineer. Fix all security vulnerabilities and hardcoded secrets in the code below.

FILE PATH: nginx.conf
FILE NAME: nginx.conf

SECURITY ISSUES TO FIX:
=== OpenGrep Security Findings ===
- [WARNING] Rule: generic.nginx.security.alias-path-traversal.alias-path-traversal
  Line 13: The alias in this location block is subject to a path traversal because the location path does not end in a path separator (e.g., '/'). To fix, add a path separator to the end of the path.
=== OpenGrep Security Findings ===
- [WARNING] Rule: generic.nginx.security.alias-path-traversal.alias-path-traversal
  Line 18: The alias in this location block is subject to a path traversal because the location path does not end in a path separator (e.g., '/'). To fix, add a path separator to the end of the path.
REPAIR INSTRUCTIONS:
1. Fix ALL listed security issues while maintai

In [ ]:
MAX_LENGTH = 4096

# Keep the original list and collect examples that fit.
train_filtered = []
dropped = 0

for row in train_data:
    # Count the full conversation, including the reference answer.
    token_ids = tokenizer.apply_chat_template(
        row["messages"],
        tokenize=True,
        add_generation_prompt=False,
        return_dict=False
    )

    if len(token_ids) <= MAX_LENGTH:
        train_filtered.append(row)
    else:
        dropped += 1

print("Training rows kept:", len(train_filtered))
print("Training rows dropped:", dropped)

Training rows kept: 443
Training rows dropped: 0


In [ ]:
# Show which GPU is connected to this runtime.
!nvidia-smi

Sun Oct 25 21:44:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Install tools for loading the model onto the GPU in 4-bit.
%pip install -q accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Store model weights in 4-bit and perform calculations in fp16.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Load the original model onto the GPU without training it.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map={"": 0},
    dtype=torch.float16
)

# Set the model to inference mode.
model.eval()

print("Base model loaded.")

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded.


In [ ]:
import math

# Select the first validation example.
row = val_data[0]

# Give the model the prompt without revealing the reference answer.
prompt_messages = row["messages"][:-1]
user_text = prompt_messages[-1]["content"]

# Extract the original file to calculate the generation budget.
marker = "FILE CONTENT:\n"
if marker not in user_text:
    raise ValueError("Could not find the FILE CONTENT marker.")

original_file = user_text.split(marker, 1)[1]
original_tokens = len(
    tokenizer.encode(original_file, add_special_tokens=False)
)
max_new_tokens = math.ceil(1.3 * original_tokens) + 100

# Format the prompt and add the assistant header for generation.
inputs = tokenizer.apply_chat_template(
    prompt_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to("cuda")

# Generate greedily, without changing the model's weights.
with torch.inference_mode():
    output = model.generate(
        **inputs,
        do_sample=False,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode only the newly generated answer, excluding the input prompt.
answer_tokens = output[0, inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)

print("Generation token budget:", max_new_tokens)
print("\nBase model answer:\n")
print(answer)

Generation token budget: 298

Base model answer:

```nginx
server {
  listen 80;
  server_name localhost;
  root /usr/share/nginx/html;
  index index.html;

  location / {
    try_files $uri $uri/ /index.html;
  }

  location /api {
    proxy_pass http://kong:8000;
    proxy_http_version 1.1;
    proxy_set_header Upgrade $http_upgrade;
    proxy_set_header Connection 'upgrade';
    proxy_set_header Host $host;
    proxy_set_header X-Real-IP $remote_addr;
    proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
    proxy_set_header X-Forwarded-Proto $scheme;
    proxy_cache_bypass $http_upgrade;
  }
}
```


In [ ]:
# Install experiment logging.
%pip install -q wandb

In [ ]:
import wandb

# Sign in using W&B's login prompt.
wandb.login(relogin=True)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: damians214 (damians214-university-of-texas-at-dallas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# Replace these with your team's W&B workspace and private project.
WANDB_ENTITY = "damians214-university-of-texas-at-dallas"
WANDB_PROJECT = "vibecheck baseline"

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name="qwen-1.5b-base-r3",
    config={
        "model": MODEL_NAME,
        "dataset": "sft_r3",
        "system": "base",
        "quantization": "4-bit NF4",
        "compute_dtype": "float16",
        "do_sample": False,
        "validation_rows": len(val_data),
        "seed": 42
    }
)

In [ ]:
import json
import math
from transformers import set_seed

set_seed(42)
model.eval()

OUTPUT_PATH = "/content/base_outputs.jsonl"

# Save each answer immediately so completed examples aren't lost on error.
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for i, row in enumerate(val_data):
        # Exclude the reference answer from the model's input.
        assert row["messages"][-1]["role"] == "assistant"
        prompt_messages = row["messages"][:-1]
        user_text = prompt_messages[-1]["content"]

        # Calculate the generation budget from the original file.
        marker = "FILE CONTENT:\n"
        if marker not in user_text:
            raise ValueError(f"Missing FILE CONTENT marker in row {i}")

        original_file = user_text.split(marker, 1)[1]
        original_tokens = len(
            tokenizer.encode(original_file, add_special_tokens=False)
        )
        budget = math.ceil(1.3 * original_tokens) + 100

        # Apply the same model chat template used earlier.
        inputs = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to("cuda")

        with torch.inference_mode():
            output = model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=budget,
                pad_token_id=tokenizer.eos_token_id
            )

        # Keep only the generated answer, without cleaning or changing it.
        answer_ids = output[0, inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(answer_ids, skip_special_tokens=True)

        record = {
            "row_index": i,
            "output": answer,
            "generated_tokens": len(answer_ids),
            "max_new_tokens": budget
        }
        f.write(json.dumps(record) + "\n")
        f.flush()

        # Log numerical progress, without uploading the generated code.
        run.log({
            "completed_examples": i + 1,
            "generated_tokens": len(answer_ids)
        })

        print(f"Completed {i + 1}/{len(val_data)}")

run.finish()
print("Saved baseline outputs to:", OUTPUT_PATH)

Completed 1/49
Completed 2/49
Completed 3/49
Completed 4/49
Completed 5/49
Completed 6/49
Completed 7/49
Completed 8/49
Completed 9/49
Completed 10/49
Completed 11/49
Completed 12/49
Completed 13/49
Completed 14/49
Completed 15/49
Completed 16/49
Completed 17/49
Completed 18/49
Completed 19/49
Completed 20/49
Completed 21/49
Completed 22/49
Completed 23/49
Completed 24/49
Completed 25/49
Completed 26/49
Completed 27/49
Completed 28/49
Completed 29/49
Completed 30/49
Completed 31/49
Completed 32/49
Completed 33/49
Completed 34/49
Completed 35/49
Completed 36/49
Completed 37/49
Completed 38/49
Completed 39/49
Completed 40/49
Completed 41/49
Completed 42/49
Completed 43/49
Completed 44/49
Completed 45/49
Completed 46/49
Completed 47/49
Completed 48/49
Completed 49/49


completed_examples,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
generated_tokens,▂▆▄▂▂▃▂▃▁▄▄▃▄▂▃▂▂▁▂▂▂▂▂▂▂▃▂▂▃▂▁▅▅▃▃█▂▂▇▂
completed_examples,49
generated_tokens,196


Saved baseline outputs to: /content/base_outputs.jsonl


In [ ]:
from google.colab import files

# Download the baseline answers to your laptop.
files.download("/content/base_outputs.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# TRL runs supervised fine-tuning.
# PEFT adds LoRA adapters.
# Datasets prepares data for the trainer.
%pip install -q trl peft datasets

In [ ]:
import trl
import peft
import transformers

print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)

TRL: 1.16.0
PEFT: 0.21.2
Transformers: 5.20.2


In [ ]:
from datasets import Dataset

# Separate each conversation into the input and the answer to learn.
training_rows = []

for row in train_filtered:
    messages = row["messages"]

    # Confirm the final message is the reference answer.
    assert messages[-1]["role"] == "assistant"

    training_rows.append({
        "prompt": messages[:-1],
        "completion": messages[-1:]
    })

train_dataset = Dataset.from_list(training_rows)

print("Training examples:", len(train_dataset))
print("Prompt roles:", [m["role"] for m in train_dataset[0]["prompt"]])
print("Completion roles:", [m["role"] for m in train_dataset[0]["completion"]])

Training examples: 443
Prompt roles: ['user']
Completion roles: ['assistant']


In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
from transformers import set_seed

set_seed(42)

# Prepare the 4-bit model for training with gradient checkpointing.
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)
model.config.use_cache = False

# Keep the assigned LoRA settings unchanged.
# Our only experiment change will be training for 2 epochs.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM"
)

# Attach the trainable adapters to the frozen base model.
model = get_peft_model(model, lora_config)

# Show how many parameters will be trained.
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
import math
from trl import SFTConfig, SFTTrainer

# Batch size 1 × accumulation 8 gives an effective batch size of 8.
total_steps = math.ceil(len(train_dataset) / 8) * 2
warmup_steps = math.ceil(total_steps * 0.03)

training_config = SFTConfig(
    output_dir="/content/vibecheck-c-2epochs",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    max_length=4096,
    packing=False,
    completion_only_loss=True,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    optim="adamw_torch",
    seed=42,
    data_seed=42,
    logging_steps=1,
    report_to="wandb",
    run_name="qwen-r3-C-2epochs",
    save_strategy="epoch",
    save_total_limit=2,
    push_to_hub=False
)

# TRL applies the tokenizer's chat template to our structured messages.
trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_dataset,
    processing_class=tokenizer
)

print("Trainer ready. Training has not started.")

Tokenizing train dataset:   0%|          | 0/443 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/443 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/443 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/443 [00:00<?, ? examples/s]

Trainer ready. Training has not started.


In [ ]:
# Inspect one prepared example without starting training.
example = trainer.train_dataset[0]
batch = trainer.data_collator([example])

ids = batch["input_ids"][0]
labels = batch["labels"][0]

# -100 labels are ignored when calculating loss.
learned_ids = ids[labels != -100]

print("Tokens excluded from loss:", (labels == -100).sum().item())
print("Tokens included in loss:", len(learned_ids))
print("\nText the model will learn to produce:\n")
print(tokenizer.decode(learned_ids, skip_special_tokens=False))

Tokens excluded from loss: 486
Tokens included in loss: 141

Text the model will learn to produce:

server {
    listen 80;
    server_name localhost;

    location / {
        proxy_pass http://app:80;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
    }

    location /swagger/ {
        alias /var/www/html/swagger.yaml/;
        default_type application/yaml;
    }

    location /swagger-ui/ {
        alias /var/www/html/public/swagger-ui/; # Assuming swagger-ui is placed here
        index index.html;
    }
}<|im_end|>



In [ ]:
# Log training separately from the completed baseline run.
run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name="qwen-r3-C-2epochs",
    save_code=False,
    config={
        "model": MODEL_NAME,
        "dataset": "sft_r3",
        "experiment": "C",
        "epochs": 2,
        "learning_rate": 2e-4,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": "all-linear",
        "max_length": 4096,
        "batch_size": 1,
        "gradient_accumulation": 8,
        "scheduler": "cosine",
        "warmup_steps": warmup_steps,
        "precision": "fp16",
        "quantization": "4-bit NF4",
        "completion_only_loss": True,
        "seed": 42,
        "training_rows": len(train_dataset),
        "dropped_rows": dropped
    }
)

In [ ]:
# Clean gradients and cast trainable parameters to float32 for fp16 training
trainer.model.zero_grad(set_to_none=True)
for parameter in trainer.model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.to(torch.float32)

# Train the LoRA adapters for two epochs.
training_result = trainer.train()

# Save the trained adapter and tokenizer locally in Colab.
ADAPTER_DIR = "/content/vibecheck-c-adapter"
trainer.save_model(ADAPTER_DIR)
tok_res = tokenizer.save_pretrained(ADAPTER_DIR)

run.finish()
print("Training complete. Adapter saved to:", ADAPTER_DIR)

Step,Training Loss
1,0.055068
2,0.043425
3,0.070256
4,0.027689
5,0.031735
6,0.022117
7,0.038700
8,0.012769
9,0.028782
10,0.013355


train/entropy,█▆▄▂▄▂▁▂▂▁▁▂▂▃▂▂▁▁▂▁▁▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇██
train/global_step,▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇███
train/grad_norm,█▆▆▅▃▂▅▆▅▄▄▄▅▃▃▂▃▄▃▄▃▂▂▃▃▂▃▂▂▂▂▃▁▁▂▂▂▃▂▂
train/learning_rate,▄██████▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁
train/loss,▆▅█▄▄▄▂▂▃▃▃▁▁▂▂▂▂▂▂▂▂▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁▂▁▁▁
train/mean_token_accuracy,▆▁▄▄██▇▇█▇███▇███▇█▇▇█████████▇███████▇▇
train/num_tokens,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇█████
total_flos,6641815747854336.0
train/entropy,0.00354
train/epoch,2


Training complete. Adapter saved to: /content/vibecheck-c-adapter


In [ ]:
from collections import Counter

# Inspect the precision of trainable weights and existing gradients.
print("Trainable weight types:")
print(Counter(
    str(p.dtype)
    for p in model.parameters()
    if p.requires_grad
))

print("\nGradient types:")
print(Counter(
    str(p.grad.dtype)
    for p in model.parameters()
    if p.requires_grad and p.grad is not None
))

print("\nTrainer precision:")
print("fp16:", trainer.args.fp16)
print("bf16:", trainer.args.bf16)
print("Accelerate:", trainer.accelerator.mixed_precision)
print("Completed optimizer steps:", trainer.state.global_step)

Trainable weight types:
Counter({'torch.float32': 392})

Gradient types:
Counter()

Trainer precision:
fp16: True
bf16: False
Accelerate: fp16
Completed optimizer steps: 112


In [ ]:
# Clear gradients left by the failed attempt.
model.zero_grad(set_to_none=True)

# Keep trainable adapter weights in float32 for fp16 mixed precision.
for parameter in model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.to(torch.float32)

print("Trainable weight types:")
print(Counter(
    str(p.dtype)
    for p in model.parameters()
    if p.requires_grad
))

Trainable weight types:
Counter({'torch.float32': 392})


In [ ]:
# Rebuild the trainer using the existing model and configuration.
# This does not attach another adapter or start training.
trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_dataset,
    processing_class=tokenizer
)

# Check whether trainer initialization changed adapter precision.
print("Trainable weight types:")
print(Counter(
    str(p.dtype)
    for p in trainer.model.parameters()
    if p.requires_grad
))

Tokenizing train dataset:   0%|          | 0/443 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/443 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/443 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/443 [00:00<?, ? examples/s]

Trainable weight types:
Counter({'torch.bfloat16': 392})


In [ ]:
# Correct adapter precision AFTER trainer initialization.
trainer.model.zero_grad(set_to_none=True)

for parameter in trainer.model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.to(torch.float32)

print(Counter(
    str(p.dtype)
    for p in trainer.model.parameters()
    if p.requires_grad
))

Counter({'torch.float32': 392})


In [ ]:
import shutil
from google.colab import files

# Package the saved adapter and tokenizer into one ZIP file.
shutil.make_archive(
    "/content/vibecheck-c-adapter",
    "zip",
    "/content/vibecheck-c-adapter"
)

files.download("/content/vibecheck-c-adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Start a separate W&B run for trained-model generation.
run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name="qwen-r3-C-2epochs-validation",
    save_code=False,
    config={
        "model": MODEL_NAME,
        "adapter": "vibecheck-c-adapter",
        "dataset": "sft_r3",
        "system": "trained-C",
        "epochs": 2,
        "do_sample": False,
        "seed": 42,
        "validation_rows": len(val_data)
    }
)

In [ ]:
import json
import math
from transformers import set_seed

set_seed(42)

# Use the trained model already in memory.
trained_model = trainer.model
trained_model.eval()
trained_model.gradient_checkpointing_disable()
trained_model.config.use_cache = True

TRAINED_OUTPUT_PATH = "/content/trained_outputs.jsonl"

with open(TRAINED_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for i, row in enumerate(val_data):
        # Hide the reference answer from the model.
        assert row["messages"][-1]["role"] == "assistant"
        prompt_messages = row["messages"][:-1]
        user_text = prompt_messages[-1]["content"]

        # Use the same generation budget as the baseline.
        marker = "FILE CONTENT:\n"
        if marker not in user_text:
            raise ValueError(f"Missing FILE CONTENT marker in row {i}")

        original_file = user_text.split(marker, 1)[1]
        original_tokens = len(
            tokenizer.encode(original_file, add_special_tokens=False)
        )
        budget = math.ceil(1.3 * original_tokens) + 100

        inputs = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to("cuda")

        # Generate greedily, using fp16 calculations on the T4.
        with torch.inference_mode(), torch.autocast(
            device_type="cuda", dtype=torch.float16
        ):
            output = trained_model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=budget,
                pad_token_id=tokenizer.eos_token_id
            )

        answer_ids = output[0, inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(answer_ids, skip_special_tokens=True)

        # Preserve the raw answer, including any formatting mistakes.
        record = {
            "row_index": i,
            "output": answer,
            "generated_tokens": len(answer_ids),
            "max_new_tokens": budget
        }
        f.write(json.dumps(record) + "\n")
        f.flush()

        run.log({
            "completed_examples": i + 1,
            "generated_tokens": len(answer_ids)
        })

        print(f"Completed {i + 1}/{len(val_data)}")

run.finish()
print("Saved trained outputs to:", TRAINED_OUTPUT_PATH)

Completed 1/49
Completed 2/49
Completed 3/49
Completed 4/49
Completed 5/49
Completed 6/49
Completed 7/49
Completed 8/49
Completed 9/49
Completed 10/49
Completed 11/49
Completed 12/49
Completed 13/49
Completed 14/49
Completed 15/49
Completed 16/49
Completed 17/49
Completed 18/49
Completed 19/49
Completed 20/49
Completed 21/49
Completed 22/49
Completed 23/49
Completed 24/49
Completed 25/49
Completed 26/49
Completed 27/49
Completed 28/49
Completed 29/49
Completed 30/49
Completed 31/49
Completed 32/49
Completed 33/49
Completed 34/49
Completed 35/49
Completed 36/49
Completed 37/49
Completed 38/49
Completed 39/49
Completed 40/49
Completed 41/49
Completed 42/49
Completed 43/49
Completed 44/49
Completed 45/49
Completed 46/49
Completed 47/49
Completed 48/49
Completed 49/49


completed_examples,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
generated_tokens,▁▅▂▁▂▂▂▂▁▃▃▂▂▂▃▂▂▁▁▁▁▂▂▂▂▂▁▂▃▂▂▁▄▄▂█▆▂▅▂
completed_examples,49
generated_tokens,193


Saved trained outputs to: /content/trained_outputs.jsonl


In [ ]:
files.download("/content/trained_outputs.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch

# Define your custom testing prompt
custom_prompt = """You are an expert security engineer. Fix all security vulnerabilities and hardcoded secrets in the code below.

FILE PATH: app.py
FILE NAME: app.py

SECURITY ISSUES TO FIX:
=== OpenGrep Security Findings ===
- [WARNING] Rule: python.lang.security.hardcoded-password
  Line 5: Hardcoded database password detected.

FILE CONTENT:
import os
import psycopg2

def connect_db():
    db_password = "super_secret_password_123"
    return psycopg2.connect(host="localhost", database="prod", user="postgres", password=db_password)
"""

# Format prompt with the Qwen template
messages = [
    {"role": "user", "content": custom_prompt}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to("cuda")

# Generate an answer using the same settings each time
def get_response():
    with torch.inference_mode(), torch.autocast(
        device_type="cuda", dtype=torch.float16
    ):
        output_ids = trained_model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=512,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)


trained_model.eval()

# Baseline: temporarily turn off the trained LoRA adapter
with trained_model.disable_adapter():
    baseline_response = get_response()

# Trained: the adapter is automatically turned back on
trained_response = get_response()

print("=== BASELINE RESPONSE ===")
print(baseline_response)

print("\n=== TRAINED RESPONSE ===")
print(trained_response)